# Burel — Colab launcher

Open language model with the HOPE / full nested-learning architecture.

**Before you start:** `Runtime → Change runtime type → GPU` (T4 is fine).

Upload `Burel.zip` to your Drive folder, then run the cells in order. The checkpoints
(`burel_best.pt` / `burel_last.pt`) are saved next to the zip: if Colab
disconnects, relaunch the training cell and it resumes on its own.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Find `Burel.zip` in Drive and unzip
Searches for the zip across all of `MyDrive`. If you put it in a specific subfolder and
want to speed things up, set `SEARCH_ROOT` to that path.

In [ ]:
import glob, os, shutil, zipfile

SEARCH_ROOT = '/content/drive/MyDrive'  # folder where you uploaded Burel.zip

hits = glob.glob(os.path.join(SEARCH_ROOT, '**', 'Burel.zip'), recursive=True)
assert hits, 'Burel.zip non trovato in Drive. Caricalo e riprova.'
zip_path = hits[0]
DRIVE_DIR = os.path.dirname(zip_path)
print('Trovato:', zip_path)
print('Cartella Drive:', DRIVE_DIR)

if os.path.exists('/content/Burel'):
    shutil.rmtree('/content/Burel')
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content')
assert os.path.exists('/content/Burel/scripts/train.py'), 'Struttura zip inattesa: manca /content/Burel/scripts/train.py'
print('Scompattato in /content/Burel')

## 3. Install the package + point checkpoints at Drive
`pip install -e .` installs Burel and its dependencies. The `.pt` files end up in
`<folder>/Burel_checkpoints`, so they survive disconnections.

In [ ]:
%cd /content/Burel
!pip -q install -e .

import yaml
cfg_path = 'configs/config.yaml'
cfg = yaml.safe_load(open(cfg_path))
cfg['train']['drive_backup_dir'] = os.path.join(DRIVE_DIR, 'Burel_checkpoints')
cfg.setdefault('data', {})['drive_cache_dir'] = os.path.join(DRIVE_DIR, 'Burel_data')
yaml.safe_dump(cfg, open(cfg_path, 'w'), sort_keys=False, allow_unicode=True)
print('Checkpoint su Drive:', cfg['train']['drive_backup_dir'])
print('Dataset cache su Drive:', cfg['data']['drive_cache_dir'])

## 4. Prepare the dataset
The active dataset is decided by `configs/config.yaml` (`data.name`): by default
`tinystories` (byte-level BPE, phase 2). The first time it downloads TinyStories from
HuggingFace, trains the BPE tokenizer and writes `train.bin`/`val.bin` + `tokenizer.json`
into `data_cache/`. It is idempotent: if the data config doesn't change, it skips the work.

In [ ]:
!python scripts/prepare_data.py

## 5. Training
Progress bar with %, ETA and loss. Automatically resumes from `burel_last.pt`
(local or Drive) if it exists. To measure speed before a long run, lower
`max_iters` in `configs/config.yaml` (e.g. 300).

In [ ]:
!python scripts/train.py

## 6. Inference — generate text

In [ ]:
!python scripts/generate.py --prompt "ROMEO:" --tokens 600 --temperature 0.7 --top_k 100

## (optional) Verify strict causality

In [ ]:
!python tests/test_causal.py